# Lab 00 · Arrume a casa antes de modelar
**Machine Learning na Prática · Dia 1 · Módulo 01: Dados & Datasets**

Neste notebook você vai fazer o que todo projeto de ML faz **antes** de treinar qualquer modelo.
Aqui **não há modelo nem métricas**: o objetivo é deixar os dados prontos. O treino começa no Lab 01, já com estes dados limpos.

| Etapa | O que fazer |
|---|---|
| 1. Inspecionar | formato, tipos, estatísticas, faltantes |
| 2. Limpar | duplicados, categorias, outliers, alvo faltante |
| 3. Explorar | histogramas, boxplot, correlação |
| 4. Preparar | features × alvo, split, one-hot, padronização e `Pipeline` sem vazamento |

Procure as células marcadas com **`# TODO`** e substitua os `___` pelo código certo.
Rode as células em ordem (`Shift + Enter`).

> **Gabarito.** Versão do professor, com todas as células resolvidas. Não compartilhe antes da atividade.

### Cola rápida de pandas
| Para… | use |
|---|---|
| ler um CSV | `pd.read_csv('arquivo.csv')` |
| formato (linhas, colunas) / tipos | `df.shape` / `df.info()` |
| estatísticas das colunas numéricas | `df.describe()` |
| faltantes por coluna | `df.isna().sum()` |
| contar valores de uma coluna | `df['col'].value_counts()` |
| achar / remover duplicados | `df.duplicated().sum()` / `df.drop_duplicates()` |
| limpar texto | `df['col'].str.strip().str.lower()` |
| remover linhas com faltante numa coluna | `df.dropna(subset=['col'])` |
| histogramas | `df.hist()` |
| correlação | `df.corr(numeric_only=True)` |
| média de um grupo | `df.groupby('col')['outra'].mean()` |

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 4)
print('Tudo pronto!')

### O dataset: desempenho de estudantes
Os dados abaixo são **sintéticos** (gerados aqui mesmo, para não depender de download), mas imitam problemas reais:
alguém digitou errado, algumas linhas foram copiadas duas vezes, o turno foi escrito de jeitos diferentes…

Rode a célula para criar o arquivo `alunos.csv`. Você **não precisa** entender o código de geração agora.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

gerar_dados().to_csv('alunos.csv', index=False)
print('alunos.csv criado!')

## 1 · Inspecionar

Carregue o arquivo e olhe as primeiras linhas.

In [ ]:
# TODO: leia o arquivo 'alunos.csv' com o pandas
df = pd.read_csv('alunos.csv')
df.head()

> 🔎 **Gabarito · o que observar**
>
> - Cada linha é um aluno; `nota_final` é o **alvo** e as outras colunas são candidatas a feature.
> - `renda_familiar` está na casa dos milhares e `horas_estudo` na casa das unidades: escalas muito diferentes (isso volta na padronização).
> - `situacao` parece só mais uma coluna, mas ela é **derivada da nota** (veja a seção 4.1).

In [ ]:
# TODO: quantas linhas e colunas? E quais os tipos de cada coluna?
print(df.shape)
df.info()

> 🔎 **Gabarito · o que observar**
>
> - **412 linhas × 8 colunas**, mas o gerador criou 400 alunos: as 12 a mais são duplicados (seção 2.1).
> - O `Non-Null Count` já denuncia os faltantes: `horas_estudo` (393), `faltas` (395) e `nota_final` (404) têm menos de 412.
> - `faltas` é contagem (inteiro), mas aparece como `float64`: no pandas, basta **um `NaN`** para a coluna inteira virar float.
> - `turno`, `monitoria` e `situacao` são texto (`object` ou `str`, conforme a versão do pandas) e vão precisar virar números.

In [ ]:
# TODO: estatísticas das colunas numéricas. Repare no máximo de horas_estudo!
df.describe()

> 🔎 **Gabarito · o que observar**
>
> - **`horas_estudo`: máximo 70**, enquanto 75% dos alunos estudam até 7,4 h. É o outlier da “vírgula esquecida”.
> - Esse outlier contamina as estatísticas: a média (6,47) já passa da mediana (6,0) e o desvio (5,76) está inflado. Depois da limpeza, o desvio cai para cerca de 2,2.
> - `nota_final` vai de 0 a 8, com média ≈ 4,1: **a maioria reprova** (só 30 dos 392 alunos finais têm nota ≥ 6).
> - `renda_familiar` tem máximo 17 060 contra mediana 3 720. É cauda longa, mas **plausível**: não é erro de digitação e não deve ser apagada.
> - `idade` (17 a 30) e `faltas` (0 a 9) estão dentro do esperado.

In [ ]:
# TODO: quantos valores faltantes há em cada coluna?
df.isna().sum()

> 🔎 **Gabarito · o que observar**
>
> - Faltantes: `horas_estudo` 19, `faltas` 17, `nota_final` 8.
> - Faltante em **feature** dá para preencher (imputar); faltante no **alvo** não, porque não há o que ensinar ao modelo com essa linha (seção 2.4).

In [ ]:
# TODO: quais valores aparecem na coluna 'turno'?
df['turno'].value_counts()

> 🔎 **Gabarito · o que observar**
>
> - Deveriam ser 3 categorias, mas aparecem **8**: `manha`, `Manhã`, `MANHÃ ` (com espaço), `Noite`, ` tarde`…
> - Para o computador são valores diferentes. Sem limpar, o one-hot criaria 8 colunas para 3 turnos de verdade.

**Para pensar:** liste os problemas que você encontrou. Quais colunas têm faltantes? O que há de estranho em `turno` e em `horas_estudo`?

*Sua resposta:* …

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** faltantes em `horas_estudo`, `faltas` e `nota_final`; `turno` escrito de vários jeitos;
> `horas_estudo` com valores absurdos (70, 68, 64); 12 linhas duplicadas; e escalas muito diferentes entre as colunas numéricas.

## 2 · Limpar

### 2.1 Duplicados

In [ ]:
# TODO: conte as linhas duplicadas e remova-as
print('duplicados:', df.duplicated().sum())
df = df.drop_duplicates()
print('linhas agora:', len(df))

> 🔎 **Gabarito · o que observar**
>
> - **12 duplicados** → 400 linhas, exatamente o número de alunos gerados.
> - Duplicado é perigoso porque a mesma linha pode cair no treino **e** no teste: o modelo “vê” a resposta antes da prova.

### 2.2 Categorias inconsistentes
Para o computador, `"manhã"`, `"Manhã"` e `"MANHÃ "` são três valores diferentes.
Padronize: tire os espaços das pontas, deixe tudo minúsculo e corrija `"manha"` sem acento.

In [ ]:
# TODO: padronize a coluna turno
df['turno'] = df['turno'].str.strip().str.lower()
df['turno'] = df['turno'].replace({'manha': 'manhã'})
df['turno'].value_counts()

> 🔎 **Gabarito · o que observar**
>
> - Agora são só 3 categorias: manhã 193, tarde 104, noite 103.
> - A ordem importa: primeiro `strip` e `lower`, depois o `replace` de `manha` → `manhã`.

### 2.3 Outliers
Ninguém estuda 70 horas *extras* por semana: provavelmente alguém esqueceu a vírgula (7,0 → 70).
Há duas saídas: **corrigir** (dividir por 10, como no slide) ou **descartar o valor** (virar `NaN`).
Como **não temos certeza** do valor certo, aqui vamos transformar em `NaN`; depois ele será preenchido como os outros faltantes.

In [ ]:
plt.boxplot(df['horas_estudo'].dropna())
plt.title('horas_estudo antes da limpeza'); plt.show()

# TODO: transforme em NaN os valores de horas_estudo acima de 24
df.loc[df['horas_estudo'] > 24, 'horas_estudo'] = np.nan
df['horas_estudo'].describe()

> 🔎 **Gabarito · o que observar**
>
> - O boxplot mostra três pontos isolados lá em cima (70, 68 e 64) e a caixa toda espremida embaixo.
> - Depois da regra, o máximo vira **15** e o desvio cai de 5,76 para 2,16.
> - **Pegadinha:** esse 15 também é um erro (o valor real era 1,5), mas 15 h por semana é plausível, então a regra `> 24` não o pega.
>   Nem todo erro é detectável por regra simples. Ele vai reaparecer no Lab 01, puxando a reta para baixo.

### 2.4 Alvo faltante
Uma linha sem `nota_final` não ensina nada: remova-a.
Os faltantes nas **features** (horas, faltas) **não** serão preenchidos agora — isso será feito dentro do `Pipeline`, usando só o treino (etapa 4).

In [ ]:
# TODO: remova as linhas sem nota_final
df = df.dropna(subset=['nota_final'])
print(len(df), 'linhas')
df.isna().sum()

> 🔎 **Gabarito · o que observar**
>
> - **392 linhas** ficam. `nota_final` não tem mais faltantes.
> - `horas_estudo` agora tem 21 faltantes (inclui os 3 outliers que viraram `NaN`) e `faltas` tem 15. Eles **continuam** faltando de propósito: quem preenche é o `Pipeline`, olhando só o treino.

## 3 · Explorar

In [ ]:
# TODO: histogramas de todas as colunas numéricas
df.hist(bins=20, figsize=(12, 6))
plt.tight_layout(); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - `renda_familiar` é **assimétrica à direita** (cauda longa de rendas altas).
> - `horas_estudo` ficou aproximadamente simétrica depois da limpeza.
> - `faltas` é discreta (só inteiros); `idade` é quase uniforme entre 17 e 30.
> - `nota_final` se concentra entre 3 e 5, e poucas passam de 6.

**Média × mediana:** compare as duas para `horas_estudo` e para `renda_familiar`. Em qual delas a diferença é maior? Por quê?

In [ ]:
for col in ['horas_estudo', 'renda_familiar']:
    # TODO: calcule a média e a mediana da coluna
    print(f"{col}: média = {df[col].mean():.1f} | mediana = {df[col].median():.1f}")

> 🔎 **Gabarito · o que observar**
>
> - `horas_estudo`: média 6,0 e mediana 6,0, praticamente iguais (distribuição simétrica).
> - `renda_familiar`: média ≈ 4 175 e mediana 3 690. A cauda de rendas altas **puxa a média para cima**; a mediana não liga para ela.
> - É por isso que, no `Pipeline`, os faltantes são preenchidos com a **mediana**.

In [ ]:
# TODO: gráfico de dispersão horas_estudo × nota_final
plt.scatter(df['horas_estudo'], df['nota_final'], alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - Tendência clara de **subida**: mais horas, nota maior em média.
> - Mas há muita dispersão: alunos com as mesmas horas têm notas bem diferentes. Horas sozinhas não explicam tudo.
> - Repare num ponto solto lá à direita, em 15 h com nota baixa: é o erro que a regra não pegou.

In [ ]:
# TODO: correlação entre as colunas numéricas
df.corr(numeric_only=True).round(2)

> 🔎 **Gabarito · o que observar**
>
> - `horas_estudo` × nota: **+0,62** (a mais forte). `faltas` × nota: **−0,31** (mais faltas, nota menor).
> - `idade` (0,03) e `renda_familiar` (−0,07) ≈ 0: praticamente **nenhuma relação linear** com a nota.
> - As features quase não se correlacionam entre si, o que é bom para a regressão.
> - A correlação só mede relação **linear** entre colunas numéricas: `turno` e `monitoria` nem aparecem aqui.

In [ ]:
# TODO: nota média por turno e por monitoria
df.groupby(['turno', 'monitoria'])['nota_final'].mean().round(2)

> 🔎 **Gabarito · o que observar**
>
> - Em **todos** os turnos, quem faz monitoria tem nota média maior (+0,4 a +0,75 ponto).
> - O turno da **noite** tem as menores médias (3,53 sem e 3,90 com monitoria).
> - Ou seja: as categorias carregam informação, e o one-hot é o jeito de entregá-las ao modelo.

**Para pensar:** quais features parecem ajudar a prever a nota? Alguma parece não ter relação nenhuma?
Lembre: correlação não é causalidade.

*Sua resposta:* …

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** `horas_estudo` (a mais forte), `faltas`, `monitoria` e `turno` ajudam; `idade` e `renda_familiar` parecem não ter relação.
> Correlação não é causa: a monitoria pode atrair justamente os alunos mais dedicados, por exemplo.

## 4 · Preparar sem vazamento
### 4.1 Features × alvo — e a coluna que “conta a resposta”
O alvo é `nota_final`. A coluna `situacao` (aprovado/reprovado) é **calculada a partir da nota**: ela só existe *depois* que a nota sai.
Usá-la como feature seria **vazamento de dados** — o modelo “colaria” a resposta. Veja como ela entrega a nota:

In [ ]:
# TODO: menor e maior nota em cada situação
df.groupby('situacao')['nota_final'].agg(['min', 'max'])

> 🔎 **Gabarito · o que observar**
>
> - Aprovado: notas de **6,0 a 8,0**. Reprovado: de **0,0 a 5,9**. A separação é perfeita, sem nenhuma sobreposição.
> - Com `situacao` como feature, o modelo descobriria a regra “nota ≥ 6” e pareceria ótimo, mas na vida real essa coluna **só existe depois** da nota. É vazamento.

In [ ]:
alvo = 'nota_final'
# TODO: escolha as features (todas menos o alvo e a coluna que vaza)
X = df.drop(columns=[alvo, 'situacao'])
y = df[alvo]

num = ['horas_estudo', 'faltas', 'idade', 'renda_familiar']
cat = ['turno', 'monitoria']
X.head()

> 🔎 **Gabarito · o que observar**
>
> - Sobram **6 features**: 4 numéricas e 2 categóricas.

### 4.2 Separar treino e teste **antes** de qualquer transformação
Aqui fazemos só treino (80%) e teste (20%). A “validação” do slide será feita com **validação cruzada** dentro do treino, a partir do Lab 01.

In [ ]:
# TODO: 80% treino, 20% teste, random_state=42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_tr.shape, X_te.shape)

> 🔎 **Gabarito · o que observar**
>
> - **313 linhas de treino e 79 de teste** (80/20 de 392).
> - O `random_state=42` faz todo mundo da turma ter a mesma divisão, então os números batem entre os colegas.

### 4.3 O preparo em um `Pipeline`
Cada peça tem um papel:
- `SimpleImputer(strategy='median')` — preenche faltantes com a **mediana** (robusta a outliers, como vimos na EDA);
- `StandardScaler()` — padroniza: `z = (x − μ) / σ`;
- `OneHotEncoder(handle_unknown='ignore')` — uma coluna 0/1 por categoria; se o teste trouxer uma categoria que o treino nunca viu, ela vira só zeros em vez de dar erro;
- `ColumnTransformer` — aplica um tratamento diferente para cada **tipo** de coluna;
- `Pipeline` — encadeia os passos e garante a ordem: tudo é **ajustado (`fit`) só no treino**.

In [ ]:
preparo = ColumnTransformer([
    # TODO: complete com SimpleImputer(strategy='median') e StandardScaler()
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median')), ('padronizar', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
])

# TODO: ajuste no treino e transforme o treino; depois só TRANSFORME o teste
Z_tr = preparo.fit_transform(X_tr)
Z_te = preparo.transform(X_te)

colunas = preparo.get_feature_names_out()
pd.DataFrame(Z_tr, columns=colunas).head()

> 🔎 **Gabarito · o que observar**
>
> - Agora são **9 colunas**: 4 numéricas padronizadas + 3 de turno + 2 de monitoria.
> - As numéricas ficam em torno de 0 (ex.: −2,18 = bem abaixo da média do treino).
> - Em cada linha, há exatamente **um 1** entre as colunas de turno e um entre as de monitoria.
> - Não sobrou nenhum `NaN`: o imputador preencheu tudo com a mediana do treino.

### 4.4 O que o `fit` aprendeu — e de onde
O `StandardScaler` guarda a média ($\mu$) e o desvio ($\sigma$) que aprendeu no `fit`.
Ajustado **só no treino**, as colunas do treino ficam com média 0 e desvio 1; as do teste ficam **perto**, mas não exatamente — afinal, o teste é “novo”.

In [ ]:
n_num = len(num)
pd.DataFrame({
    'média treino':  Z_tr[:, :n_num].mean(axis=0),
    'desvio treino': Z_tr[:, :n_num].std(axis=0),
    'média teste':   Z_te[:, :n_num].mean(axis=0),
}, index=num).round(3)

> 🔎 **Gabarito · o que observar**
>
> - No treino: média **0** e desvio **1** em todas as colunas, por construção.
> - No teste: médias perto de 0, mas não exatas (algo como 0,12; 0,06; 0,06; −0,14). Isso é o **esperado**: o teste foi transformado com o μ e o σ do treino.
> - Se o teste também saísse com média 0 exata, seria sinal de que ele participou do ajuste.

Agora o jeito **errado**: ajustar o scaler com o dataset inteiro (treino + teste) antes de separar.
O $\mu$ aprendido muda — ou seja, **informação do teste entrou no ajuste**. Com poucos dados ou estatísticas mais sensíveis, isso infla o resultado.

In [ ]:
mu_certo = preparo.named_transformers_['num'].named_steps['padronizar'].mean_
# TODO: ajuste um StandardScaler com TODO o X numérico (o jeito errado) e pegue as médias
mu_errado = StandardScaler().fit(X[num].fillna(X[num].median())).mean_

pd.DataFrame({'μ só treino (certo)': mu_certo, 'μ treino + teste (errado)': mu_errado}, index=num).round(2)

> 🔎 **Gabarito · o que observar**
>
> - As médias mudam pouco (ex.: renda 4 239 × 4 175), mas **mudam**: o μ “errado” carrega informação do teste.
> - Aqui o efeito é pequeno porque há bastante dado e a média é estável. O problema é de princípio:
>   com poucos dados, transformações mais complexas ou seleção de features, o vazamento infla a nota do modelo.

### 4.5 Salvar os dados limpos
O Lab 01 começa exatamente daqui, com esta versão limpa.

In [ ]:
df.to_csv('alunos_limpo.csv', index=False)
print(df.shape, '→ alunos_limpo.csv')

> 🔎 **Gabarito · o que observar**
>
> - `(392, 8)`: o `alunos_limpo.csv` é o ponto de partida do Lab 01.
> - Ele ainda tem `NaN` nas features e ainda tem `situacao`. Isso é de propósito: quem cuida disso é o `Pipeline` de cada modelo.

## 5 · Desafios extras (se sobrar tempo)
### 5.1 Quem é mais parecido com você? (escala em código)
Compare o aluno mais parecido com o primeiro aluno do treino usando a **distância euclidiana**,
com e sem padronização. Use só as colunas numéricas sem faltantes.

In [ ]:
cols = ['faltas', 'idade', 'renda_familiar']
A = X_tr[cols].dropna()
voce, outros = A.iloc[0], A.iloc[1:]

def mais_parecido(dados, ref):
    dist = np.sqrt(((dados - ref) ** 2).sum(axis=1))
    return dist.idxmin()

# TODO: padronize com a média e o desvio de A
Az = (A - A.mean()) / A.std()

print('você:', voce.to_dict())
print('sem padronizar →', A.loc[mais_parecido(outros, voce)].to_dict())
print('padronizando   →', A.loc[mais_parecido(Az.iloc[1:], Az.iloc[0])].to_dict())

> 🔎 **Gabarito · o que observar**
>
> - **Sem padronizar**, o “mais parecido” tem renda quase igual (8 150 × 8 170) mas idade **17 contra 22**: só a renda decidiu.
> - **Padronizando**, aparece alguém com a **mesma idade** (22) e renda próxima (8 850).
> - Sem padronizar, uma diferença de 20 reais pesa o mesmo que 20 anos, porque a distância soma números crus.

**Para pensar:** sem padronizar, qual coluna decidiu quem é “parecido”? Por quê?

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** `renda_familiar`, porque está na casa dos milhares e domina a soma das diferenças ao quadrado.

### 5.2 Label encoding × one-hot
Transforme `turno` em números das duas formas e compare.

In [ ]:
label = df['turno'].map({'noite': 0, 'manhã': 1, 'tarde': 2})
# TODO: one-hot da coluna turno com o pandas
onehot = pd.get_dummies(df['turno'], prefix='turno', dtype=int)
pd.concat([df['turno'], label.rename('label'), onehot], axis=1).head()

> 🔎 **Gabarito · o que observar**
>
> - O label encoding cria uma **ordem falsa**: noite (0) < manhã (1) < tarde (2), e a “distância” de noite até tarde vira o dobro de noite até manhã.
> - O one-hot dá uma coluna 0/1 por turno, sem ordem e sem distâncias inventadas.

**Para pensar:** no label encoding, “tarde” (2) é o dobro de “manhã” (1)? Que problema isso causa num modelo que faz contas com esses números?

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** não. Um modelo linear multiplicaria o turno por um peso, como se “tarde” tivesse o dobro do efeito de “manhã”, o que não faz sentido.
> Label encoding só é adequado para categorias com ordem real (ex.: fundamental < médio < superior), ou em árvores, que não fazem contas com o valor.

## Para discutir
1. Transformar os outliers em `NaN` ou dividir por 10: qual decisão você tomaria? O que muda nos dados?
2. Por que preencher os faltantes **dentro** do pipeline, e não antes do `train_test_split`?
3. Que outra coluna, num problema real, poderia “contar a resposta” como a `situacao`?

> 🔎 **Gabarito · o que observar**
>
> **Pontos para a discussão:**
> 1. Dividir por 10 recupera o valor, se o palpite estiver certo; virar `NaN` é mais conservador e deixa o imputador preencher com a mediana. Note que o 15 escapou das duas abordagens.
> 2. Porque a mediana calculada antes do split usaria dados do teste: o mesmo vazamento da seção 4.4.
> 3. Exemplos: “cancelou o plano” para prever churn, data de alta para prever tempo de internação, “nota de recuperação” para prever reprovação.